# RobotMath2030 — Neural Operators / DeepONet (Ch.13)

Learn initial condition → trajectory map; amortize simulation rollouts.

Full chapter: [13_neural_operators](../chapters/13_neural_operators/)

In [ ]:
import time
from pathlib import Path

try:
    import google.colab  # noqa: F401
    !git clone -q https://github.com/rsasaki0109/RobotMath2030.git 2>/dev/null || true
    %cd RobotMath2030
except ImportError:
    root = Path.cwd()
    if not (root / 'robotmath').exists() and (root.parent / 'robotmath').exists():
        %cd ..

!pip install -q -e ".[torch]"
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from robotmath.neural_operators import OperatorTrainConfig, make_operator_dataset, operator_trajectory_mse, train_deeponet
from robotmath.physics.mass_spring import MassSpringParams, simulate

train_u, times, train_y = make_operator_dataset(n_samples=256, seed=0)
test_u, _, test_y = make_operator_dataset(n_samples=32, seed=1)
model, losses = train_deeponet(train_u, times, train_y, OperatorTrainConfig(epochs=80, seed=0))
test_mse = operator_trajectory_mse(model, test_u, times, test_y)
print(f'Operator test MSE: {test_mse:.6f}')

params = MassSpringParams()
steps = len(times) - 1
n_query = 200
t0 = time.perf_counter()
for u in test_u[:n_query]:
    simulate(params, float(u[0]), float(u[1]), steps)
sim_time = time.perf_counter() - t0
t0 = time.perf_counter()
model.predict_trajectory(test_u[:n_query], times)
op_time = time.perf_counter() - t0
print(f'Batch {n_query} — simulator: {sim_time:.3f}s, operator: {op_time:.3f}s')

u_demo = test_u[0]
y_pred = model.predict_trajectory(u_demo[None, :], times)[0]
y_true = test_y[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogy(losses, 'C0-')
axes[0].set_xlabel('epoch')
axes[0].set_title('DeepONet training')
axes[0].grid(True, alpha=0.3)
axes[1].plot(times, y_true, 'k-', label='truth')
axes[1].plot(times, y_pred, 'C0-o', markersize=3, label='operator')
axes[1].legend()
axes[1].set_xlabel('time [s]')
axes[1].set_title('Held-out trajectory')
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()